In [1]:

import scanpy as sc
import cell2location
import numpy as np
import matplotlib.pyplot as plt
results_folder = '/nfs/users/nfs_d/dp26/nfs_storage/Megagut/cell2location_run/'

# create paths and names to results folders for reference regression and cell2location models
ref_run_name = f'{results_folder}/result_hgca_scRNAseq/reference_signatures_hgca_celltype_v1'
run_name = f'{results_folder}/result_hgca_visium/c2l_map_hgca_ct1_only'

adata_file = f"{ref_run_name}/sc.h5ad"
adata_ref = sc.read_h5ad(adata_file)
mod = cell2location.models.RegressionModel.load(f"{ref_run_name}", adata_ref)

inf_aver = adata_ref.varm['means_per_cluster_mu_fg'][[f'means_per_cluster_mu_fg_{i}' for i in adata_ref.uns['mod']['factor_names']]].copy()
inf_aver.columns = adata_ref.uns['mod']['factor_names']
inf_aver.iloc[0:5, 0:5]

/nfs/team205/dp26/conda_env/cell2location_cuda118_torch22/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


INFO     File                                                                                                      
         /nfs/users/nfs_d/dp26/nfs_storage/Megagut/cell2location_run//result_hgca_scRNAseq/reference_signatures_hgc
         a_celltype_v1/model.pt already downloaded                                                                 


/nfs/team205/dp26/conda_env/cell2location_cuda118_torch22/lib/python3.10/site-packages/scvi/data/fields/_dataframe_field.py:224: UserWarning: Category 478 in adata.obs['_scvi_batch'] has fewer than 3 cells. Models may not train properly.
  new_mapping = _make_column_categorical(
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
/nfs/team205/dp26/conda_env/cell2location_cuda118_torch22/lib/python3.10/site-packages/lightning/pytorch/trainer/configuration_validator.py:72: You passed in a `val_dataloader` but have no `validation_step`. Skipping val loop.
You are using a CUDA device ('NVIDIA A100-SXM4-80GB MIG 1c.2g.20gb') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html

Epoch 1/76:   1%|▏         | 1/76 [00:00<00:29,  2.50it/s, v_num=1]

`Trainer.fit` stopped: `max_steps=1` reached.


Epoch 1/76:   1%|▏         | 1/76 [00:00<00:31,  2.40it/s, v_num=1]


,Adipocytes,Angiogenic Pericytes,Arteriolar Endothelial,BEST4 Colonocytes,BEST4 Enterocytes
ENSG00000000003,0.118526,0.050715,0.243448,0.748824,0.456467
ENSG00000000005,0.008852,0.001201,0.015152,0.002518,0.000646
ENSG00000000419,0.358764,0.184689,0.278142,0.420856,0.333026
ENSG00000000457,0.026554,0.040440,0.038663,0.124367,0.124952
ENSG00000000460,0.011616,0.011896,0.008062,0.023325,0.012197


In [2]:
import os
os.makedirs(run_name,exist_ok=True)

In [3]:
filenames = [
    'spaceranger311_count_36090_WSSS_A_GUTsp9518708_1_GRCh38-2020-A',
 # 'spaceranger311_count_35963_6330STDY9479161_GRCh38-2020-A',
 'spaceranger311_count_36090_WSSS_A_GUTsp9518709_1_GRCh38-2020-A',
 # 'spaceranger311_count_35963_6330STDY9479160_GRCh38-2020-A',
 # 'spaceranger311_count_35963_6330STDY9479163_GRCh38-2020-A',
 'spaceranger311_count_36090_WSSS_A_GUTsp9518707_1_GRCh38-2020-A',
 # 'spaceranger311_count_35963_6330STDY9479164_GRCh38-2020-A',
 # 'spaceranger311_count_35963_6330STDY9479162_GRCh38-2020-A',
 'spaceranger311_count_36090_WSSS_A_GUTsp9518706_1_GRCh38-2020-A',
 # 'spaceranger311_count_35963_6330STDY9479165_GRCh38-2020-A',
 # 'spaceranger311_count_35963_6330STDY9479159_GRCh38-2020-A',
 # 'spaceranger311_count_35963_6330STDY9479158_GRCh38-2020-A'
]

In [4]:
BASE_PATH = '/nfs/users/nfs_d/dp26/nfs_storage/Megagut/dataset/visium/'

In [5]:
adatas = []
for filename in filenames:
    try:
        tmp = sc.read_visium(BASE_PATH + filename)
    except:
        import shutil
        src = BASE_PATH + filename + '/spatial/tissue_positions.csv'
        dst = BASE_PATH + filename + '/spatial/tissue_positions_list.csv'
        shutil.copyfile(src, dst)
        tmp = sc.read_visium(BASE_PATH + filename)
    tmp.var = tmp.var.reset_index().rename(columns={'index': 'gene_symbols'}).set_index('gene_ids')
    adatas.append(tmp)

/nfs/team205/dp26/conda_env/cell2location_cuda118_torch22/lib/python3.10/site-packages/anndata/_core/anndata.py:1820: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/nfs/team205/dp26/conda_env/cell2location_cuda118_torch22/lib/python3.10/site-packages/anndata/_core/anndata.py:1820: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/nfs/team205/dp26/conda_env/cell2location_cuda118_torch22/lib/python3.10/site-packages/anndata/_core/anndata.py:1820: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/nfs/team205/dp26/conda_env/cell2location_cuda118_torch22/lib/python3.10/site-packages/anndata/_core/anndata.py:1820: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("va

In [6]:
adata_concat = sc.concat(adatas,label="batch",keys=filenames,uns_merge="unique",index_unique='-batch-')

In [7]:

intersect = np.intersect1d(adata_concat.var_names, inf_aver.index)
adata_concat = adata_concat[:, intersect].copy()
inf_aver = inf_aver.loc[intersect, :].copy()


In [ ]:
cell2location.models.Cell2location.setup_anndata(adata=adata_concat, batch_key="batch")

# create and train the model
mod = cell2location.models.Cell2location(
    adata_concat, cell_state_df=inf_aver, 
    # the expected average cell abundance: tissue-dependent 
    # hyper-prior which can be estimated from paired histology:
    N_cells_per_location=30,
    # hyperparameter controlling normalisation of
    # within-experiment variation in RNA detection:
    detection_alpha=20
) 
mod.view_anndata_setup()
mod.train(max_epochs=30000, 
          # train using full data (batch_size=None)
          batch_size=None, 
          # use all data points in training because 
          # we need to estimate cell abundance at all locations
          train_size=1,
          accelerator='cuda',
         )


Anndata setup with scvi-tools version 1.1.5.

Setup via `Cell2location.setup_anndata` with arguments:

{
│   'layer': None,
│   'batch_key': 'batch',
│   'labels_key': None,
│   'categorical_covariate_keys': None,
│   'continuous_covariate_keys': None
}

         Summary Statistics         
┏━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━┓
┃     Summary Stat Key     ┃ Value ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━┩
│         n_batch          │   4   │
│         n_cells          │ 8867  │
│ n_extra_categorical_covs │   0   │
│ n_extra_continuous_covs  │   0   │
│         n_labels         │   1   │
│          n_vars          │ 17318 │
└──────────────────────────┴───────┘

               Data Registry                
┏━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Registry Key ┃    scvi-tools Location    ┃
┡━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│      X       │          adata.X          │
│    batch     │ adata.obs['_scvi_batch']  │
│    ind_x     │   adata.obs['_indices']   │
│    labels    │ adata.obs['_scvi_labels'] │
└──────────────┴───────────────────────────┘

                                            batch State Registry                                             
┏━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━┓
┃  Source Location   ┃                           Categories                           ┃ scvi-tools Encoding ┃
┡━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━┩
│ adata.obs['batch'] │ spaceranger311_count_36090_WSSS_A_GUTsp9518708_1_GRCh38-2020-A │          0          │
│                    │ spaceranger311_count_36090_WSSS_A_GUTsp9518709_1_GRCh38-2020-A │          1          │
│                    │ spaceranger311_count_36090_WSSS_A_GUTsp9518707_1_GRCh38-2020-A │          2          │
│                    │ spaceranger311_count_36090_WSSS_A_GUTsp9518706_1_GRCh38-2020-A │          3          │
└────────────────────┴────────────────────────────────────────────────────────────────┴─────────────────────┘

                     labels State Registry                      
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━┓
┃      Source Location      ┃ Categories ┃ scvi-tools Encoding ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━┩
│ adata.obs['_scvi_labels'] │     0      │          0          │
└───────────────────────────┴────────────┴─────────────────────┘

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
/nfs/team205/dp26/conda_env/cell2location_cuda118_torch22/lib/python3.10/site-packages/lightning/pytorch/trainer/configuration_validator.py:72: You passed in a `val_dataloader` but have no `validation_step`. Skipping val loop.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [MIG-GPU-06ece157-3e57-e120-e6a8-59e2375d1036/3/0]
/nfs/team205/dp26/conda_env/cell2location_cuda118_torch22/lib/python3.10/site-packages/lightning/pytorch/loops/fit_loop.py:293: The number of training batches (1) is smaller than the logging interval Trainer(log_every_n_steps=10). Set a lower value for log_every_n_steps if you want to see logs for the training epoch.


Epoch 725/30000:   2%|▏         | 724/30000 [03:58<2:41:58,  3.01it/s, v_num=1, elbo_train=3.68e+7]

In [ ]:
adata_concat.write_h5ad(f"{results_folder}/gut_visium_hgca_samples.h5ad")

In [ ]:
# In this section, we export the estimated cell abundance (summary of the posterior distribution).
adata_concat =  mod.export_posterior(
    adata_concat, sample_kwargs={'num_samples': 1000, 'batch_size': mod.adata.n_obs}
)


In [ ]:
adata_concat

In [ ]:
os.makedirs(f"{run_name}/mod_level3",exist_ok=True)
mod.save(f"{run_name}/mod_level3", overwrite=True)

In [ ]:

# Save anndata object with results
adata_file = f"{run_name}/Gut_Visium_healthy_adult_4_samples.h5ad"
adata_concat.write(adata_file)
adata_file

In [14]:
print(f"{run_name}/Gut_Visium_healthy_adult_4_samples.h5ad")

/nfs/users/nfs_d/dp26/nfs_storage/Megagut/cell2location_run//result_hgca_visium/c2l_map_hgca_ct1_only/Gut_Visium_healthy_adult_4_samples.h5ad


In [15]:
!ls /nfs/users/nfs_d/dp26/nfs_storage/Megagut/cell2location_run//result_hgca_visium/c2l_map_hgca_ct1_only/Gut_Visium_healthy_adult_4_samples.h5ad

/nfs/users/nfs_d/dp26/nfs_storage/Megagut/cell2location_run//result_hgca_visium/c2l_map_hgca_ct1_only/Gut_Visium_healthy_adult_4_samples.h5ad


/nfs/team205/dp26/conda_env/cell2location_cuda118_torch22/lib/python3.10/pty.py:89: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  pid, fd = os.forkpty()
